# **Import Libraries**

In [1]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer #convert text to vectors
import faiss #get similar vectors to provided instruction (need to be install using pip install faiss-cpu)
from transformers import AutoTokenizer, AutoModelForCausalLM #auto token to convert words to IDs, Automodel for Qwen model
import torch #LLM model working using pytorch

# **Import data set**

In [3]:
file_path = "/content/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
df = pd.read_csv(file_path)
initial_shape = df.shape
print("Shape:", df.shape)
df.head()

Shape: (26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


# **Text clean function**

In [4]:
def clean_text_nlp(text, remove_numbers=False):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    #drop place holders as {{Order Number}}
    text = re.sub(r"\{\{.*?\}\}", " ", text)
    #remove or keep numbers if needed
    if remove_numbers:
        text = re.sub(r"[^a-z\s]", " ", text)
    else:
        text = re.sub(r"[^a-z0-9\s]", " ", text)
    #keep only one space (drop first and last spaces)
    text = re.sub(r"\s+", " ", text).strip()
    return text

#We won't clear stop words or verbs ... etc since it is better for more robustness system (they are important in customer support)

# **Add cleaned instruction & response to the data frame**

In [5]:
#apply clean
columns_to_clean = ["flags", "instruction", "category", "intent", "response"]

for col in columns_to_clean:
    df[f"{col}_clean"] = df[col].apply(lambda x: clean_text_nlp(x, remove_numbers=True))

df[['instruction', 'instruction_clean', 'response', 'response_clean']].head()

,instruction,instruction_clean,response,response_clean
0,question about cancelling order {{Order Number}},question about cancelling order,I've understood you have a question regarding ...,i ve understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,i have a question about cancelling oorder,I've been informed that you have a question ab...,i ve been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},i need help cancelling puchase,I can sense that you're seeking assistance wit...,i can sense that you re seeking assistance wit...
3,I need to cancel purchase {{Order Number}},i need to cancel purchase,I understood that you need assistance with can...,i understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",i cannot afford this order cancel purchase,I'm sensitive to the fact that you're facing f...,i m sensitive to the fact that you re facing f...


In [6]:
# remove rows with empty important fields

required_clean_columns = [
    "instruction_clean",
    "response_clean",
    "intent_clean",
    "category_clean"
]

# null row condition
mask = (df[required_clean_columns] != "").all(axis=1)

df = df[mask].reset_index(drop=True)

print("Shape after removing empty rows:", df.shape)

percentage = ((initial_shape[0] - df.shape[0]) / initial_shape[0]) * 100
print("Percentage of empty rows : ", percentage, "%")

Shape after removing empty rows: (26872, 10)
Percentage of empty rows :  0.0 %


# **Embedding Model**

In [7]:
#document
df["retrieval_text"] = (
    "passage: " +
    df["instruction_clean"].fillna("") + " " +
    df["intent_clean"].fillna("") + " " +
    df["category_clean"].fillna("") + " " +
    df["flags_clean"].fillna("")
).str.strip()

In [8]:
#input query
queries = df["retrieval_text"].fillna("").astype(str).tolist() #type = string and convert to list for embedding vectors

print("Number of texts:", len(queries))
print("First 5 sample texts:")
for i, text in enumerate(queries[:5]):
    print(f"{i+1}. {text}")

Number of texts: 26872
First 5 sample texts:
1. passage: question about cancelling order cancel order order b
2. passage: i have a question about cancelling oorder cancel order order bqz
3. passage: i need help cancelling puchase cancel order order blqz
4. passage: i need to cancel purchase cancel order order bl
5. passage: i cannot afford this order cancel purchase cancel order order bceln


In [9]:
#embedding model
embedding_model_name = "BAAI/bge-base-en-v1.5"
embedding_model = SentenceTransformer(embedding_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
#Convert instructions to vectors

# use GPU if available (important for large datasets)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = embedding_model.to(device)

embeddings = embedding_model.encode(
    queries,
    batch_size=64,                 # important for speed with large data
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True      # important for better similarity (especially with BGE)
)

print("Embeddings shape:", embeddings.shape) #embedding dimension depends on model

Batches:   0%|          | 0/420 [00:00<?, ?it/s]

Embeddings shape: (26872, 768)


In [11]:
embedding_dim = embeddings.shape[1]
print("Embedding dimension:", embedding_dim)

Embedding dimension: 768


In [12]:
#reduce size to float 32 (better for FAISS lib and to reduce memory & time)
embeddings = embeddings.astype("float32", copy=False)
print("Embeddings dtype:", embeddings.dtype)

Embeddings dtype: float32


# **FAISS Index to search similarity**

In [13]:
#Build FAISS index

embedding_dim = embeddings.shape[1]

#use Inner Product (cosine similarity since embeddings are normalized)
index = faiss.IndexFlatIP(embedding_dim)

index.add(embeddings)

print("Number of vectors inside FAISS index:", index.ntotal)

#now our FAISS contains all queries vectors (it will compare vectors based on cosine similarity)

Number of vectors inside FAISS index: 26872


# **Build Retrieval model (RAG Part 1)**

In [14]:
#Now time to retrieval function
def retrieve_similar_queries(query, k=8): #return nearest similar questions from database
    #clean user query for retrieval
    query_clean = clean_text_nlp(query, remove_numbers=True)

    #for BGE models, it is better to prefix the query
    formatted_query = f"query: {query_clean}"

    #convert query to vector
    query_embedding = embedding_model.encode(
        [formatted_query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    query_embedding = query_embedding.astype("float32", copy=False)

    #search in FAISS for similars
    scores, indices = index.search(query_embedding, k)

    #get results to query from data set
    results = df.iloc[indices[0]][[
        "instruction",
        "response",
        "instruction_clean",
        "response_clean",
        "intent",
        "intent_clean",
        "category",
        "category_clean",
        "flags",
        "flags_clean"
    ]].copy()

    #add score to evaluate
    results["score"] = scores[0]
    results["query"] = query
    results["query_clean"] = query_clean

    #sort results from best to worst (higher score is better with IndexFlatIP)
    results = results.sort_values(by="score", ascending=False).reset_index(drop=True)

    return results

In [15]:
#Test a sample
query = "i wanna cancel my order"
results = retrieve_similar_queries(query, k=10)
results
#excellent result

,instruction,response,instruction_clean,response_clean,intent,intent_clean,category,category_clean,flags,flags_clean,score,query,query_clean
0,I want to cancel purchase {{Order Number}},I can see that you want to cancel purchase {{O...,i want to cancel purchase,i can see that you want to cancel purchase ple...,cancel_order,cancel order,ORDER,order,BL,bl,0.857098,i wanna cancel my order,i wanna cancel my order
1,I want to cancel purchase {{Order Number}},I've picked up that you would like to cancel y...,i want to cancel purchase,i ve picked up that you would like to cancel y...,cancel_order,cancel order,ORDER,order,BL,bl,0.857098,i wanna cancel my order,i wanna cancel my order
2,help me to cancel my last goddamn purchase,I've picked up that you're feeling frustrated ...,help me to cancel my last goddamn purchase,i ve picked up that you re feeling frustrated ...,cancel_order,cancel order,ORDER,order,BLWZ,blwz,0.856026,i wanna cancel my order,i wanna cancel my order
3,help me to cancel purchase {{Order Number}},"I understood, you need assistance with canceli...",help me to cancel purchase,i understood you need assistance with cancelin...,cancel_order,cancel order,ORDER,order,BL,bl,0.851382,i wanna cancel my order,i wanna cancel my order
4,help me to cancel purchase {{Order Number}},I'm aware that you need assistance with cancel...,help me to cancel purchase,i m aware that you need assistance with cancel...,cancel_order,cancel order,ORDER,order,BL,bl,0.851382,i wanna cancel my order,i wanna cancel my order
5,help me to cancel purchase {{Order Number}},I've noticed that you need assistance with can...,help me to cancel purchase,i ve noticed that you need assistance with can...,cancel_order,cancel order,ORDER,order,BL,bl,0.851382,i wanna cancel my order,i wanna cancel my order
6,help me to cancel purchase {{Order Number}},For sure! I understand that you need assistanc...,help me to cancel purchase,for sure i understand that you need assistance...,cancel_order,cancel order,ORDER,order,BL,bl,0.851382,i wanna cancel my order,i wanna cancel my order
7,help me to cancel purchase {{Order Number}},I perceive that you're seeking assistance with...,help me to cancel purchase,i perceive that you re seeking assistance with...,cancel_order,cancel order,ORDER,order,BL,bl,0.851382,i wanna cancel my order,i wanna cancel my order
8,"I no longer want order {{Order Number}}, how d...","I realized, you no longer want to proceed with...",i no longer want order how do i cancel it,i realized you no longer want to proceed with ...,cancel_order,cancel order,ORDER,order,BCILN,bciln,0.849011,i wanna cancel my order,i wanna cancel my order
9,help me cancelling purchase {{Order Number}},"Of course, I'm here to assist you with canceli...",help me cancelling purchase,of course i m here to assist you with cancelin...,cancel_order,cancel order,ORDER,order,BL,bl,0.848563,i wanna cancel my order,i wanna cancel my order


In [16]:
#Test another samples
test_queries = [
    "refund not arrived",
    "wrong address order",
    "need support",
    "need customer support",
    "payment options"
]

for q in test_queries:
    print("-" * 80)
    print("Query:", q)

    results = retrieve_similar_queries(q, k=5)

    print(results[[
        "instruction",
        "response",
        "intent",
        "category",
        "score"
    ]].head(3))

    print()

--------------------------------------------------------------------------------
Query: refund not arrived
                                         instruction  \
0  how long until I receive areimbursement of money?   
1                 how long until i get my money back   
2                                   status of refund   

                                            response        intent category  \
0  I understand your concern regarding the timefr...    get_refund   REFUND   
1  I can sense that you're eager to know how long...    get_refund   REFUND   
2  I comprehend that you're seeking an update on ...  track_refund   REFUND   

      score  
0  0.770617  
1  0.768006  
2  0.767899  

--------------------------------------------------------------------------------
Query: wrong address order
                                         instruction  \
0  I have submitted a wrong address, help me edit...   
1  i submitted an wrong address i need help to ed...   
2          i have 

End of RAG Part 1 (R: Retrieval)
We got all related data to the given query and returned them.

# **Now let's build RAG Part 2 - A: Augmentation**

In [17]:
#A is taking the output from R then convert them to prompt context to prepare LLM Model for answer

#small helper function to make text cleaner for generation without destroying meaning
def prepare_text_for_generation(text):
    if pd.isna(text):
        return ""

    text = str(text).strip()

    #replace placeholders with readable generic fields
    text = re.sub(r"\{\{.*?\}\}", "your account details", text)

    #keep text natural but remove extra spaces/new lines
    text = re.sub(r"\s+", " ", text).strip()

    return text


#context function
def build_context(retrieved_results, max_examples=8):
    context_parts = []

    #take only the top results needed for context
    selected_results = retrieved_results.head(max_examples)

    for i, (_, row) in enumerate(selected_results.iterrows()):
        instruction_text = prepare_text_for_generation(row["instruction"])
        intent_text = prepare_text_for_generation(row["intent"])
        category_text = prepare_text_for_generation(row["category"])
        response_text = prepare_text_for_generation(row["response"])

        part = (
            f"Example {i+1}:\n"
            f"Customer Query: {instruction_text}\n"
            f"Intent: {intent_text}\n"
            f"Category: {category_text}\n"
            f"Support Response:\n{response_text}\n"
        )
        context_parts.append(part)

    return "\n\n".join(context_parts)

In [18]:
#Test a sample
sample_results = retrieve_similar_queries("how can i cancel my order", k=8)

print("=" * 100)
print("Retrieved Results (with scores):\n")

for i, (_, row) in enumerate(sample_results.head(3).iterrows(), start=1):
    print(f"Example {i} (Score: {row['score']:.4f})")
    print("Customer Query:", row["instruction"])
    print("Intent:", row["intent"])
    print("Category:", row["category"])
    print("Response:")
    print(row["response"])
    print("-" * 100)

# build context after inspection
context_text = build_context(sample_results, max_examples=6)

print("\n\n" + "=" * 100)
print("FINAL CONTEXT SENT TO MODEL:\n")
print(context_text)

Retrieved Results (with scores):

Example 1 (Score: 0.8321)
Customer Query: how do i cancel order {{Order Number}}
Intent: cancel_order
Category: ORDER
Response:
Sure! I'm here to help you cancel order {{Order Number}}. To initiate the cancellation process, please follow these steps:

1. Open your {{Online Company Portal Info}} and sign in to your account.
2. Locate the "{{Online Order Interaction}}" or "{{Online Order Interaction}}" section on the platform.
3. Find the specific order with the order number {{Order Number}} and click on it.
4. Look for the option labeled "{{Online Order Interaction}}" associated with your purchase and select it.
5. You may be asked to provide a reason for canceling the order. Please provide the necessary information and proceed with the cancellation.

If you encounter any difficulties or have further questions during the cancellation process, please don't hesitate to reach out to our customer support team. They are available {{Customer Support Hours}} a

End of Part 2

# **LLM Model**

In [19]:
#upload LLM Model

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

#load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

#load model (use GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded on:", device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda


# **RAG Part 3 (G: Generation)**

In [20]:
def generate_answer_with_llm(prompt, max_new_tokens=200):
    messages = [
        {
            "role": "system",
            "content": "You are a professional customer support assistant. Produce clear, accurate, well-organized, and helpful responses."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    #convert chat messages to model text format
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_text,
        return_tensors="pt", #return pytorch tensors
        truncation=True, #crop when exceed max input tokens
        max_length=2048
    )

    #move inputs to the same device as model
    device = next(llm_model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = llm_model.generate( #to generate the output
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    #decode only newly generated tokens, not the full prompt
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True) #convert vector to string for reading

    return generated_text.strip()

End of Part 3

# **LLM Prompt Function**

In [21]:
def build_prompt(user_query, context):
    prompt = f"""
You are a highly professional customer support assistant.

Your job is to answer the customer's question using the retrieved support examples in the context below.

Important instructions:
1. Read all retrieved examples carefully before answering.
2. Use the context as your main source of truth.
3. Extract the most relevant steps, policies, and actions from the examples.
4. Write the final answer in clear, natural, professional English.
5. Use complete sentences, correct grammar, correct spelling, and proper punctuation.
6. Organize the answer in the best possible way for customer support.
7. If the answer contains steps or actions, present them as numbered steps or bullet points when appropriate.
8. If multiple retrieved examples provide complementary details, combine them into one coherent final answer.
9. If the wording in the context is messy, simplified, or repetitive, rewrite it into polished customer-support language.
10. If placeholders appear in the context, replace them with natural generic wording such as "your order number", "your account", or "your payment details" whenever appropriate.
11. Do not copy fragmented text, broken phrases, raw template artifacts, or placeholder-like tokens into the final answer.
12. Do not mention the retrieved examples, the context, or that you are using support data.
13. Do not give a weak refusal such as saying you do not have enough information unless the context is truly unusable.
14. Even if the context is incomplete, provide the most helpful and professional answer possible based on the available information.
15. Do not invent company policies that are not supported by the retrieved examples. When details are missing, give a careful and practical answer based only on the available context.

Context:
{context}

Customer Question:
{user_query}

Final Answer:
"""
    return prompt.strip()

# **General Questions Prompt**

In [22]:
def generate_direct_answer(user_query, max_new_tokens=200):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Answer clearly, naturally, and in polished English."
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    device = next(llm_model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return generated_text.strip()

# **Check if question related to company**

In [23]:
def is_company_related_query(user_query):
    company_keywords = [
    "order", "refund", "return", "shipping", "shipment", "delivery",
    "payment", "billing", "invoice", "cancel", "support",
    "account", "purchase", "subscription", "plan", "address",
    "policy", "rule", "escalation", "vip", "fees", "verification"
]

    query_clean = clean_text_nlp(user_query, remove_numbers=True)
    return any(keyword in query_clean for keyword in company_keywords)

# **Prompt in case of missing company data in RAG**

In [24]:
def build_safe_support_prompt(user_query):
    prompt = f"""
You are a professional customer support assistant.

The user's question appears to be related to company support, but the retrieved support knowledge is not strong enough to confirm exact company-specific details.

Your task:
1. Provide a careful and helpful reply.
2. Do not invent company-specific rules, policies, timelines, or guarantees.
3. Give only general guidance that is safe and reasonable.
4. Tell the customer that exact details should be confirmed with customer support if needed.
5. Use clear, polite, professional English.
6. Organize the answer with bullets or steps if helpful.

Customer Question:
{user_query}

Final Answer:
"""
    return prompt.strip()

# **Case Question not found in RAG and related to company**

In [25]:
SUPPORT_PHONE = "+20-XXX-XXXXXXX"
SUPPORT_EMAIL = "support@company.com"
SUPPORT_WEBSITE = "https://yourcompany.com/support"

def build_escalation_message():
    if not SUPPORT_PHONE or not SUPPORT_EMAIL:
        return (
            "I'm sorry, but I cannot confirm the exact company-specific answer for this request at the moment. "
            f"Please contact customer support through our official page:\n{SUPPORT_WEBSITE}"
        )

    return (
        "I'm sorry, but I cannot confirm the exact company-specific answer for this request at the moment. "
        "Please contact customer support directly:\n\n"
        f"Phone: {SUPPORT_PHONE}\n"
        f"Email: {SUPPORT_EMAIL}"
    )

# **RAG Function (Combine of R-A-G Functions**)

In [26]:
def answer_user_query(user_query, first_k=4, second_k=8, max_examples=2, max_new_tokens=200):
    user_query = str(user_query).strip()

    # Step 1: retrieve with fallback
    retrieval_data = retrieve_with_fallback(
        user_query=user_query,
        first_k=first_k,
        second_k=second_k
    )

    retrieved_results = retrieval_data["retrieved_results"]
    rag_eval = retrieval_data["rag_eval"]
    retrieval_round = retrieval_data["retrieval_round"]

    # Step 2: check if query is company-related
    company_related = is_company_related_query(user_query)

    # CASE 1: RAG
    # Use RAG whenever confidence is medium or high
    high_risk_keywords = ["policy", "rule", "exact", "legal", "guarantee"]
    is_high_risk = any(word in clean_text_nlp(user_query) for word in high_risk_keywords)
    if rag_eval["confidence"] in ["high", "medium"] and not is_high_risk:
        context = build_context(retrieved_results, max_examples=max_examples)
        prompt = build_prompt(user_query, context)
        answer = generate_answer_with_llm(prompt, max_new_tokens=max_new_tokens)

        return {
            "query": user_query,
            "mode": "RAG",
            "top_score": rag_eval["top_score"],
            "avg_top3_score": rag_eval["avg_top3_score"],
            "confidence": rag_eval["confidence"],
            "retrieval_round": retrieval_round,
            "retrieved_results": retrieved_results,
            "context": context,
            "prompt": prompt,
            "answer": answer
        }

    # CASE 2: DIRECT
    # If not company-related, answer directly from model knowledge
    if not company_related:
        answer = generate_direct_answer(user_query, max_new_tokens=max_new_tokens)

        return {
            "query": user_query,
            "mode": "DIRECT",
            "top_score": rag_eval["top_score"],
            "avg_top3_score": rag_eval["avg_top3_score"],
            "confidence": rag_eval["confidence"],
            "retrieval_round": retrieval_round,
            "retrieved_results": retrieved_results,
            "context": "",
            "prompt": user_query,
            "answer": answer
        }

    # If company-related and retrieval is low, do one cautious answer first
    if rag_eval["confidence"] == "low":
        safe_prompt = build_safe_support_prompt(user_query)
        answer = generate_answer_with_llm(safe_prompt, max_new_tokens=max_new_tokens)

        return {
            "query": user_query,
            "mode": "SAFE_DIRECT",
            "top_score": rag_eval["top_score"],
            "avg_top3_score": rag_eval["avg_top3_score"],
            "confidence": rag_eval["confidence"],
            "retrieval_round": retrieval_round,
            "retrieved_results": retrieved_results,
            "context": "",
            "prompt": safe_prompt,
            "answer": answer
        }

    # CASE 3: ESCALATE_TO_SUPPORT
    # Only when company-related and retrieval is very low even after wider retrieval
    answer = build_escalation_message()

    return {
        "query": user_query,
        "mode": "ESCALATE_TO_SUPPORT",
        "top_score": rag_eval["top_score"],
        "avg_top3_score": rag_eval["avg_top3_score"],
        "confidence": rag_eval["confidence"],
        "retrieval_round": retrieval_round,
        "retrieved_results": retrieved_results,
        "context": "",
        "prompt": "",
        "answer": answer
    }

# **Evaluate RAG score**

In [27]:
def evaluate_rag_confidence(retrieved_results):
    if retrieved_results is None or len(retrieved_results) == 0:
        return {
            "top_score": 0.0,
            "avg_top3_score": 0.0,
            "confidence": "very_low"
        }

    top_score = float(retrieved_results["score"].iloc[0])
    avg_top3_score = float(retrieved_results["score"].head(3).mean())

    if top_score >= 0.82 and avg_top3_score >= 0.76:
        confidence = "high"
    elif top_score >= 0.74 and avg_top3_score >= 0.68:
        confidence = "medium"
    elif top_score >= 0.66:
        confidence = "low"
    else:
        confidence = "very_low"

    return {
        "top_score": top_score,
        "avg_top3_score": avg_top3_score,
        "confidence": confidence
    }

# **Check before skip the RAG by more results k = 4, 8**

In [28]:
def retrieve_with_fallback(user_query, first_k=4, second_k=8):
    # first retrieval
    results_1 = retrieve_similar_queries(user_query, k=first_k)
    eval_1 = evaluate_rag_confidence(results_1)

    # if good enough, stop here
    if eval_1["confidence"] in ["high", "medium"]:
        return {
            "retrieved_results": results_1,
            "rag_eval": eval_1,
            "retrieval_round": 1
        }

    # second wider retrieval to avoid missing paraphrased questions
    results_2 = retrieve_similar_queries(user_query, k=second_k)
    eval_2 = evaluate_rag_confidence(results_2)

    # keep the better one based on top score then avg score
    if (
        eval_2["top_score"] > eval_1["top_score"]
        or (
            eval_2["top_score"] == eval_1["top_score"]
            and eval_2["avg_top3_score"] > eval_1["avg_top3_score"]
        )
    ):
        return {
            "retrieved_results": results_2,
            "rag_eval": eval_2,
            "retrieval_round": 2
        }

    return {
        "retrieved_results": results_1,
        "rag_eval": eval_1,
        "retrieval_round": 1
    }

# **Test the prototype by a sample**

In [29]:
result = answer_user_query(
    user_query="cancel my order please",
    first_k=4,
    second_k=8,
    max_examples=2,
    max_new_tokens=200
)

print("Mode:", result["mode"])
print("Answer:")
print(result["answer"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Mode: RAG
Answer:
To cancel your order, please follow these steps:

1. Log in to your account using your login credentials.
2. Navigate to the 'my orders' or 'order history' section within your account.
3. Locate the specific order containing the product you wish to cancel.
4. Click on the 'cancel order' button or similar action to initiate the cancellation process.
5. Review any prompts or confirmations required before completing the cancellation.

If you need further assistance or have any questions along the way, our dedicated support team is available to help you at your account details.


In [30]:
#More samples
test_queries = [
    # inside RAG domain
    "cancel my order please",
    "where is my refund",
    "what payment methods do you accept",
    "can i change my shipping address",
    "how do i contact customer support",

    # general / outside RAG domain
    "what is the capital of Egypt and number of citizens?",
    "write a short motivational quote for work",
    "what time is it now in Cairo?",
    "explain machine learning in simple words",
    "tell me a short joke",

    # company-related but may be outside available RAG knowledge
    "i need the exact company rule for a billing issue",
    "what is your internal escalation policy for VIP customers?",
    "can your company waive late payment fees in special cases?",
    "what is the company rule if my order is partially delivered?",
    "what is your policy for suspicious payment verification?"
]

for i, q in enumerate(test_queries, start=1):
    result = answer_user_query(
        user_query=q,
        first_k=4,
        second_k=8,
        max_examples=2,
        max_new_tokens=200
    )

    print("=" * 100)
    print(f"Test {i}")
    print("=" * 100)
    print("Question:")
    print(q)
    print()
    print("Mode:")
    print(result["mode"])
    print()
    print("Answer:")
    print(result["answer"])
    print("\n" + "-" * 100 + "\n")

Test 1
Question:
cancel my order please

Mode:
RAG

Answer:
To cancel your order, please follow these steps:

1. Log in to your account using your login credentials.
2. Navigate to the 'my orders' or 'order history' section within your account.
3. Locate the specific order containing the product you wish to cancel.
4. Click on the 'cancel order' button or similar action to initiate the cancellation process.
5. Review any prompts or confirmations required before completing the cancellation.

If you need further assistance or have any questions along the way, our dedicated support team is available to help you at your account details.

----------------------------------------------------------------------------------------------------

Test 2
Question:
where is my refund

Mode:
RAG

Answer:
To check the status of your refund, follow these steps:

1. Log in to your account on our website.
2. Navigate to the "My Orders" or "My Purchases" section.
3. Locate your specific order or transactio